# Aula 08 — NumPy: Ufuncs, Broadcasting e Estatística

**Semana 4 | 50 min | Referências: VanderPlas, *Python Data Science Handbook*, Cap. 2 · McKinney, *Python for Data Analysis*, Caps. 4 e 12**

## 🎯 Objetivos de aprendizagem

Ao final desta aula você será capaz de:

- Usar ufuncs (`+ − * /`, `np.exp`, `np.maximum`, `np.cumprod`) sobre matrizes inteiras sem laço;
- Aplicar as regras de broadcasting para combinar uma matriz 6 × 36 com um vetor de 36 fatores;
- Construir o **deflator do IPCA** a partir das variações mensais (BCB-SGS 433) e converter faturamento nominal em reais de jul/2026;
- Resumir matrizes com `mean`, `std`, `sum`, `min`, `max` e `quantile` controlando o eixo (`axis=0` vs `axis=1`);
- Filtrar com máscaras booleanas: meses acima da meta, filiais abaixo do piso de R$ 100 mil;
- Medir com `%timeit` o custo do laço explícito vs a versão vetorizada.

## 1. Recriando o faturamento e as ufuncs elementares

**Intuição.** A mesma rede da Aula 07: 6 filiais de Goiás, 36 meses de faturamento
nominal (ago/2023 → jul/2026). Recriamos a matriz com a mesma semente (`42`), então
os números são **exatamente** os da aula passada. Agora a pergunta muda: em vez de
*olhar* a matriz, vamos *transformá-la* — reajustar, acumular, comparar — sempre sem
laço.

In [1]:
# Setup — imports e padrões visuais usados na aula
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

def brl(x):
    """Formata um número como moeda brasileira: R$ 1.234,56."""
    return f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

np.random.seed(42)  # reprodutibilidade quando houver aleatoriedade

In [2]:
# Matriz de faturamento: mesma geração da Aula 07 (semente 42 -> números idênticos)
filiais = ["Goiânia Campinas", "Goiânia Bueno", "Anápolis", "Rio Verde", "Jataí", "Catalão"]
rng = np.random.default_rng(42)

base = np.array([1_350_000, 920_000, 680_000, 430_000, 190_000, 88_000], dtype=float)
cresc = np.array([0.004, 0.003, 0.005, 0.006, 0.008, 0.015])
m = np.arange(36)

tendencia = (1 + cresc[:, None]) ** m
sazonalidade = 1 + 0.06 * ((m % 12) >= 10)
ruido = rng.normal(0, 0.05, size=(6, 36))
fat = np.round(base[:, None] * tendencia * sazonalidade * (1 + ruido), 2)

# rótulos de mês para ler resultados ("08/2023" ... "07/2026")
mes_num = np.arange(36)
ano = 2023 + (7 + mes_num) // 12
mes_do_ano = ((7 + mes_num) % 12) + 1
meses = np.array([f"{m2:02d}/{a}" for m2, a in zip(mes_do_ano, ano)])

print("filiais:", len(filiais), "| shape:", fat.shape, "| dtype:", fat.dtype)

filiais: 6 | shape: (6, 36) | dtype: float64


In [3]:
# Ufuncs elementares: reajuste de 5% e piso por filial com np.maximum
reajustado = fat * 1.05                       # +5% em todos os 216 valores
piso_custo = np.full((6, 36), 100_000.0)      # piso de referência por mês-filial
com_piso = np.maximum(fat, piso_custo)        # máximo ELEMENTO a elemento

print("mês 0, Goiânia Campinas: nominal", brl(fat[0, 0]), "-> reajustado", brl(reajustado[0, 0]))
print("Catalão, mês 0 (88 mil):", brl(fat[5, 0]), "-> com piso", brl(com_piso[5, 0]))

mês 0, Goiânia Campinas: nominal R$ 1.370.568,40 -> reajustado R$ 1.439.096,82
Catalão, mês 0 (88 mil): R$ 93.746,41 -> com piso R$ 100.000,00


**Leitura do resultado.** `fat * 1.05` reajustou os 216 valores de uma vez; e
`np.maximum(fat, piso)` aplicou o piso onde o faturamento ficou abaixo de R$ 100 mil
(a Catalão, filial nova, no início da série). Ufunc = operação elemento a elemento,
sem laço visível.

## 2. O IPCA real: do CSV ao deflator

**Intuição.** "Quanto cresceu de verdade?" exige converter cada mês a reais de hoje
(data-base: jul/2026, o último mês da janela). O IBGE/BCB não guarda "reais de hoje";
guarda **variações mensais** (%) — o IPCA. A construção tem dois passos:
(1) **acumular** as variações num índice $I_t = \prod (1+i_k)$;
(2) para trazer o mês $t$ para a data-base (jul/2026), **multiplicar** pelo fator
$\text{fator}_t = I_{36}/I_t$: como os preços subiram entre $t$ e a data-base, R$ 1,00
do mês $t$ equivale a mais de 1 real de hoje. O fator do último mês é 1; os anteriores,
> 1.

In [4]:
# Lendo o IPCA mensal do BCB (SGS 433): sep ';' e decimal ',' | últimos 36 meses -> índice
from pathlib import Path

DATA = Path("../data/csv")   # notebook mora em semana_04/
ipca = pd.read_csv(DATA / "ipca_mensal.csv", sep=";", decimal=",")
ipca["data"] = pd.to_datetime(ipca["data"], dayfirst=True)   # dd/mm/aaaa do BCB
ipca = ipca.sort_values("data").reset_index(drop=True)

ipca_36 = ipca.tail(36).reset_index(drop=True)        # ago/2023 -> jul/2026
infl_mensal = ipca_36["valor"].to_numpy() / 100.0     # % -> decimal (ex.: 0,52% -> 0,0052)
meses_ipca = ipca_36["data"].dt.strftime("%m/%Y").to_numpy()

indice = np.cumprod(1.0 + infl_mensal)                # I_t (base = início da janela)
print("inflação acumulada na janela:", f"{(indice[-1] - 1):.2%}")
print("primeiro/último mês da janela:", meses_ipca[0], "->", meses_ipca[-1])

inflação acumulada na janela: 14.84%
primeiro/último mês da janela: 08/2023 -> 07/2026


In [5]:
# Fator de deflação para reais de jul/2026: fator_t = I_36 / I_t
fator = indice[-1] / indice
print("fator ago/2023 (mês 0):", round(fator[0], 4))
print("fator jul/2026 (mês 35):", round(fator[-1], 4), "(== 1.0: é a data-base)")

fator ago/2023 (mês 0): 1.1458
fator jul/2026 (mês 35): 1.0 (== 1.0: é a data-base)


**Leitura do resultado.** O fator de ago/2023 é ≈ **1,146**: para trazer um valor de
lá para reais de jul/2026, multiplique por 1,146 — ou seja, os preços subiram ~14,8% no
período (o `1.00` do último mês confirma que a data-base está certa). Checar
`fator[-1] == 1.0` é a verificação de sanidade que evita deflacionar "para trás".

## 3. Broadcasting: uma linha para deflacionar 216 valores

**Intuição.** `fat` tem shape `(6, 36)`; `fator` tem shape `(36,)`. As regras de
broadcasting comparam as dimensões **da direita para a esquerda**: `36` com `36` —
iguais; o array menor não tem a dimensão das linhas, então é tratado como `(1, 36)` e
**esticado** sobre as 6 filiais. Resultado: cada coluna de `fat` é **multiplicada** pelo
fator **do seu mês** (multiplicar, não dividir: o fator dos meses passados é > 1, pois
os preços subiram até a data-base) — sem replicar memória e sem laço.

In [6]:
# A linha da aula: levar todos os valores a reais de jul/2026 (broadcasting)
real = fat * fator                # (6, 36) * (36,) -> NumPy completa (36,) como (1, 36)

print("shape:", real.shape)
print("Goiânia Campinas ago/2023: nominal", brl(fat[0, 0]), "| real", brl(real[0, 0]))

# crescimento da rede (totais mensais) nominal vs real
tot_nominal = fat.sum(axis=0)     # soma as 6 filiais -> um total por mês (36,)
tot_real = real.sum(axis=0)
cresc_nom = tot_nominal[-1] / tot_nominal[0] - 1
cresc_real = tot_real[-1] / tot_real[0] - 1
print(f"crescimento ago/2023 -> jul/2026: nominal {cresc_nom:.1%} | real {cresc_real:.1%}")

shape: (6, 36)
Goiânia Campinas ago/2023: nominal R$ 1.370.568,40 | real R$ 1.570.403,57
crescimento ago/2023 -> jul/2026: nominal 26.0% | real 9.9%


**Leitura do resultado.** A rede cresceu ~26% em valor nominal, mas só ~**10%** em
reais de jul/2026 — a inflação comeu a diferença. Esse é o número que muda a conversa
com o dono: "faturamos mais" e "vendemos mais" não são a mesma frase. Veja ainda que a
série real é **mais plana** que a nominal: deflacionar comprime o passado, pois os
valores antigos ganham fator > 1.

## 4. Agregações e o parâmetro `axis`

**Intuição.** Resumir a matriz é o trabalho do dia a dia: total por mês, total por
filial, média da rede. O parâmetro `axis` escolhe **a direção que a operação atravessa
e elimina**: `axis=0` atravessa as linhas (colapsa as filiais, sobra o mês);
`axis=1` atravessa as colunas (colapsa os meses, sobra a filial). Decore pelo
**shape do resultado**.

In [7]:
# As quatro perguntas clássicas: total geral, por mês, por filial, ranking
print("total geral 36 meses :", brl(fat.sum()))
print("total por mês        : shape", fat.sum(axis=0).shape, "-> ex. jul/2026:", brl(fat.sum(axis=0)[-1]))
print("total por filial     :")
for nome, total in zip(filiais, fat.sum(axis=1)):
    print(f"   {nome:<17} {brl(total)}")

total geral 36 meses : R$ 144.378.877,50
total por mês        : shape (36,) -> ex. jul/2026: R$ 4.608.240,63
total por filial     :
   Goiânia Campinas  R$ 52.925.323,37
   Goiânia Bueno     R$ 35.213.396,92
   Anápolis          R$ 26.713.167,05
   Rio Verde         R$ 17.382.376,52
   Jataí             R$ 7.940.740,39
   Catalão           R$ 4.203.873,25


In [8]:
# Média, desvio-padrão, mínimo, máximo e quantis POR FILIAL (axis=1)
resumo = pd.DataFrame({
    "média": fat.mean(axis=1),
    "desvio": fat.std(axis=1),
    "mínimo": fat.min(axis=1),
    "máximo": fat.max(axis=1),
    "q25": np.quantile(fat, 0.25, axis=1),
    "q75": np.quantile(fat, 0.75, axis=1),
}, index=filiais)
resumo.round(0).astype(int)

,média,desvio,mínimo,máximo,q25,q75
Goiânia Campinas,1470148,111555,1237915,1738473,1392370,1538335
Goiânia Bueno,978150,50621,883997,1095337,949833,1012947
Anápolis,742032,44231,648739,843727,713577,762107
Rio Verde,482844,43100,395220,593027,451860,510372
Jataí,220576,23713,182277,269986,200896,239678
Catalão,116774,18536,88797,164948,102316,129080


**Leitura do resultado.** O desvio-padrão por filial (axis=1) cresce com o tamanho da
filial — Goiânia Campinas oscila mais **em reais** (e em percentual é a mais estável:
0,05 de ruído para todas). `np.quantile(fat, 0.25, axis=1)` dá o quartil de cada
filial; a distância q75−q25 (amplitude interquartílica) resume a dispersão sem se
abalar por outliers de safra.

## 5. Máscaras booleanas: filtrar sem laço

**Intuição.** As perguntas gerenciais são sempre condicionais: *quais meses a rede
bateu a meta? quais filiais ficaram abaixo do piso?* A receita tem dois passos:
(1) a comparação produz um array `True/False` (a **máscara**);
(2) usar a máscara como índice filtra. Combinar condições exige `&`, `|`, `~` com
**parênteses** — `&` tem precedência sobre `>`.

In [9]:
# Meses em que a REDE bateu a meta de R$ 3,8 milhões
meta_rede = 3_800_000.0
tot_nominal = fat.sum(axis=0)
mascara_meta = tot_nominal > meta_rede          # máscara 1-D: True = mês acima da meta

print("meses acima da meta:", mascara_meta.sum(), "de", len(mascara_meta))
print("primeiros meses acima:", meses[mascara_meta][:4])

meses acima da meta: 27 de 36
primeiros meses acima: ['11/2023' '06/2024' '07/2024' '08/2024']


In [10]:
# Filiais com faturamento mensal abaixo de R$ 100 mil: onde, quantas vezes
mascara_piso = fat < 100_000                    # máscara 2-D 6 × 36
por_filial = mascara_piso.sum(axis=1)           # meses abaixo do piso, por filial

for nome, n in zip(filiais, por_filial):
    print(f"   {nome:<17} {n} meses abaixo do piso")
print()
print("meses da Catalão abaixo do piso:", meses[mascara_piso[5]])

   Goiânia Campinas  0 meses abaixo do piso
   Goiânia Bueno     0 meses abaixo do piso
   Anápolis          0 meses abaixo do piso
   Rio Verde         0 meses abaixo do piso
   Jataí             0 meses abaixo do piso
   Catalão           8 meses abaixo do piso

meses da Catalão abaixo do piso: ['08/2023' '09/2023' '10/2023' '11/2023' '12/2023' '02/2024' '03/2024'
 '04/2024']


In [11]:
# Combinação de condições: meses em que a rede bateu a meta E a Catalão ficou < 100 mil
catalao_baixo = fat[5] < 100_000                                  # (36,)
ambas = (tot_nominal > meta_rede) & catalao_baixo                 # & exige parênteses!
print("meses com rede acima da meta E Catalão < R$ 100 mil:", ambas.sum())
meses[ambas]

meses com rede acima da meta E Catalão < R$ 100 mil: 1


array(['11/2023'], dtype='<U7')

**Leitura do resultado.** A Catalão — filial nova — é a única com meses abaixo do
piso (8 meses, mínimo ≈ R$ 88,8 mil). A máscara combinada mostra um trade-off real:
há meses em que a rede **bate** a meta agregada enquanto a Catalão segue fraca —
crescimento concentrado nas filiais maduras.

## 6. Loop vs vetorizado: a conta final

**Intuição.** A deflação via broadcasting foi uma linha. A "mesma" tarefa com laço
explícito (o jeito pré-NumPy) custa dezenas de vezes mais tempo. Vamos medir as duas
sobre a matriz inteira 6 × 36 — e conferir que os resultados são idênticos com
`np.allclose` (a forma profissional de validar a versão rápida).

In [12]:
# Versão laço (2 fors) vs vetorizada: medir e conferir que o resultado é idêntico
def real_com_laco(mat, fac):
    out = np.empty_like(mat)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            out[i, j] = mat[i, j] * fac[j]
    return out

print("laço:")
%timeit real_com_laco(fat, fator)

print("vetorizado:")
%timeit fat * fator

np.allclose(real_com_laco(fat, fator), real)   # True: mesmos números, só a velocidade muda

laço:


30.5 μs ± 295 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
vetorizado:


558 ns ± 0.886 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


True

**Leitura do resultado.** Mesmo em uma matriz pequena (216 valores), o vetorizado
ganha por dezenas de vezes — e escala: com uma matriz de milhões de valores, a
diferença vira "segundos vs minutos". O `np.allclose(...) == True` fecha o argumento:
não há exatidão perdida, só velocidade e legibilidade ganhas. No laço a operação era
`fat[i, j] * fator[j]`; no vetorizado, `fat * fator`.

## 📝 Exercícios

Tente resolver **antes** de abrir a célula `# SOLUÇÃO N`. Use `fat`, `real`, `fator`,
`tot_nominal`, `meses` e `filiais` das seções anteriores.

### Exercício 1 — Deflacionar na mão (para entender o broadcasting)

Deflacione **apenas a linha da Goiânia Campinas** (índice 0) com um laço `for` sobre
os 36 fatores (`goiania_real_loop`) e, em seguida, com uma única operação vetorizada
(`goiania_real_vec`). Confirme com `np.allclose` que os resultados são iguais.

In [13]:
# EXERCÍCIO 1 — seu código aqui



In [14]:
# SOLUÇÃO 1
goiania_real_loop = np.empty(36)
for j in range(36):
    goiania_real_loop[j] = fat[0, j] * fator[j]   # um fator por mês, laço explícito

goiania_real_vec = fat[0] * fator                 # ufunc + broadcasting em 1-D
print("resultados idênticos?", np.allclose(goiania_real_loop, goiania_real_vec))
print("média real da Goiânia Campinas:", brl(goiania_real_vec.mean()))

resultados idênticos? True
média real da Goiânia Campinas: R$ 1.572.178,08


### Exercício 2 — Meta anual de 2025

Calcule o **total nominal do ano de 2025** (os 12 meses de índice 17 a 28 na janela
ago/2023 → jul/2026). Confira o calendário com `meses[17]` e `meses[28]` antes de
somar.

In [15]:
# EXERCÍCIO 2 — seu código aqui



In [16]:
# SOLUÇÃO 2
print("janela pedida:", meses[17], "->", meses[28])   # jan/2025 -> dez/2025
fat_2025 = fat[:, 17:29]                              # 12 colunas, todas as filiais
total_2025 = fat_2025.sum()
brl(total_2025)

janela pedida: 01/2025 -> 12/2025


'R$ 48.930.058,69'

### Exercício 3 — O mês mais fraco da rede

Entre os 36 totais mensais (`fat.sum(axis=0)`), encontre o **índice do pior mês** com
`argmin` e exiba o mês (`meses[...]`) e o valor com `brl()`.

In [17]:
# EXERCÍCIO 3 — seu código aqui



In [18]:
# SOLUÇÃO 3
tot = fat.sum(axis=0)
pior_mes = tot.argmin()               # POSIÇÃO do mínimo, não o valor
print("pior mês da rede:", meses[pior_mes], "-", brl(tot[pior_mes]))

pior mês da rede: 09/2023 - R$ 3.542.367,21


### Exercício 4 — Filiais persistentemente abaixo do piso

Usando `axis=1`, descubra **quais filiais** têm pelo menos um mês com faturamento
abaixo de R$ 100 mil (dica: `(fat < 100_000).any(axis=1)` devolve um booleano por
filial). Liste os nomes das filiais selecionadas.

In [19]:
# EXERCÍCIO 4 — seu código aqui



In [20]:
# SOLUÇÃO 4
pior_mes_filial = fat.min(axis=1)                    # menor mês de cada filial
abaixo_do_piso = fat[(fat < 100_000).any(axis=1)]    # linhas (filiais) selecionadas
filiais_abaixo = [f for f, p in zip(filiais, pior_mes_filial) if p < 100_000]

print("matriz recortada:", abaixo_do_piso.shape)
print("filiais com algum mês < R$ 100 mil:", filiais_abaixo)

matriz recortada: (1, 36)
filiais com algum mês < R$ 100 mil: ['Catalão']


## 📌 Resumo & para casa

- Ufuncs aplicam a operação a **todo o array** de uma vez: `fat * 1.05`,
  `np.maximum(fat, piso)`, `np.cumprod(1 + infl)`.
- Broadcasting: dimensões comparadas da direita para a esquerda; `(6, 36) / (36,)`
  funciona porque `(36,)` é tratado como `(1, 36)` e esticado sobre as filiais.
- Deflator do IPCA: variações mensais → índice `cumprod` → `fator = indice[-1]/indice`;
  verificação de sanidade: `fator[-1] == 1.0`. A rede cresceu ~26% nominal, ~10% real.
- `axis` = a direção que a agregação **elimina**: `axis=0` → um número por mês;
  `axis=1` → um número por filial.
- Máscaras: `(cond).sum()` conta, `(cond).mean()` dá proporção; combinações com
  `&`/`|`/`~` **sempre parentetizadas**.
- Laço vs vetorizado: mesmo resultado (`np.allclose`), dezenas de vezes de diferença
  em tempo — e o vetorizado não tem bug de índice.
- **Para casa**: reproduzir o deflator com a janela dos **últimos 24 meses** em vez de
  36 (mude `.tail(36)` para `.tail(24)`) e redeflacionar; observe como o fator do
  primeiro mês cai.
- **Próxima aula (A09)**: o pandas organiza essas mesmas operações com rótulos
  (DataFrames) — adeus, índice de mês "17".

Referências (KB):
- `kb/02_vanderplas_python_data_science_handbook/05_chapter-2-introduction-to-numpy.md` — Cap. 2.
- `kb/01_mckinney_python_for_data_analysis/06_chapter-4-numpy-basics-arrays-and-vectorized-computation.md` — Cap. 4.
- `kb/01_mckinney_python_for_data_analysis/14_chapter-12-advanced-numpy.md` — Cap. 12 (broadcasting em profundidade).